In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
from pyspark.sql import functions as F

print("[STEP 1] Reading source bronze table")
source_query = f"SELECT * FROM abd_caresync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'"
print(f"[SOURCE QUERY] \n {source_query}")

bronze_df = (
    spark.read.table(f"abd_caresync.bronze.{table_name}")
    .filter(f"landing_timestamp = '{landing_timestamp}'")
)

print(f"[STEP 1 COMPLETE] Source DataFrame prepared for table '{table_name}' with landing_timestamp '{landing_timestamp}'")
print(f"[SOURCE COLUMNS] {bronze_df.columns}")

In [0]:
print("[STEP 2] Preparing load metadata")

target_table = f"abd_caresync.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print(f"[TARGET TABLE] {target_table}")
print(f"[RAW LOAD TYPE] '{load_type}'")
print(f"[NORMALIZED LOAD TYPE] {normalized_load_type}")
print(f"[MERGE KEYS] {merge_key_list}")

if normalized_load_type == "MERGE":
    print("[MERGE VALIDATION] Validating merge keys against source columns")
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")
    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")
    print("[MERGE VALIDATION COMPLETE] All merge keys are present in source data")

# Add audit timestamp columns
updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("[STEP 2 COMPLETE] Added insert_timestamp and update_timestamp columns")
print(f"[UPDATED COLUMNS] {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

print(f"[STEP 3] Writing data to target table: {target_table} | Load Type: {normalized_load_type}")

if normalized_load_type == "FULL":
    # Overwrite the silver table completely with the new batch
    updated_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    print(f"[STEP 3 COMPLETE] Successfully performed FULL overwrite on {target_table}")

elif normalized_load_type == "APPEND":
    # Append new rows to the existing silver table
    updated_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)
    print(f"[STEP 3 COMPLETE] Successfully performed APPEND on {target_table}")

elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        # If the target table doesn't exist yet, create it via saveAsTable first
        print(f"[MERGE INFO] Target table {target_table} does not exist. Creating it with initial data.")
        updated_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)
    else:
        # Perform Delta MERGE (Upsert) based on provided merge keys
        delta_table = DeltaTable.forName(spark, target_table)
        
        # Build match condition string dynamically (e.g., target.id = source.id)
        match_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_key_list])
        print(f"[MERGE CONDITION] {match_condition}")
        
        (delta_table.alias("target")
            .merge(updated_df.alias("source"), match_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
            
    print(f"[STEP 3 COMPLETE] Successfully performed MERGE on {target_table}")

else:
    raise ValueError(f"Unsupported load_type: '{normalized_load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
print("[STEP 4] Computing processed record count")
record_count = updated_df.count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")

dbutils.notebook.exit(str(record_count))